In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Loading raw dataset

In [52]:
df = pd.read_csv("../data/raw/CIBRS_Group_A_Public_Crime_Data_20261113.csv", low_memory=False)

print(df.shape)
display(df.head())


(806935, 22)


,Incident UID,Agency,Incident Date,CIBRS Unique Offense ID,Crime Against Category,CIBRS Offense Code,CIBRS Offense Description,Victim UID,Victim Category,Victim Age,...,Beat,City,Zip Code,Census Tract,Block Address,CIBRS Status,CIBRS Status Date,Domestic Violence Incident,BCS Area,Stolen Vehicles
0,604311,SAN DIEGO,2021 Jan 01 01:25:00 AM,604311-290,Property,290,Destruction/Damage/Vandalism of Property,NaN,NaN,NaN,...,451,SAN DIEGO,92115.0,NaN,0 BLOCK DOVEFLOWER WAY,NaN,2021-12-31 00:36:32,False,NaN,NaN
1,605986,HARBOR POLICE,2021 Jan 01 08:40:00 AM,605986-26C,Property,26C,Impersonation,NaN,NaN,NaN,...,750,SAN DIEGO,92101.0,NaN,3500 BLOCK ADMIRAL BOLAND WAY,NaN,2021-12-31 01:09:21,False,NaN,NaN
2,773968,SHERIFF,2021 Jan 01 12:30:00 PM,773968-13A-797918,People,13A,Aggravated Assault,797918.0,I,36.0,...,29,SHERIFF,NaN,NaN,0 BLOCK GBDF,NaN,2021-12-31 11:29:20,False,DETENTION FACILITY,NaN
3,687369,SYCUAN TRIBAL POLICE DEPARTMENT,2021 Jan 03 01:00:00 AM,687369-23H,Property,23H,All Other Larceny,NaN,NaN,NaN,...,1,EL CAJON,92019.0,NaN,5400 BLOCK CASINO WAY,NaN,NaN,False,NaN,NaN
4,1072215,LA MESA,2021 Jan 03 04:37:00 PM,1072215-13B-1109606,People,13B,Simple Assault,1109606.0,I,21.0,...,3,LA MESA,NaN,NaN,0 BLOCK EL PASO STREET,NaN,2022-05-24 16:09:15,True,NaN,NaN


In [53]:
print(df.columns.tolist())

['Incident UID', 'Agency', 'Incident Date', 'CIBRS Unique Offense ID', 'Crime Against Category', 'CIBRS Offense Code', 'CIBRS Offense Description', 'Victim UID', 'Victim Category', 'Victim Age', 'Victim Race', 'Victim Sex', 'Beat', 'City', 'Zip Code', 'Census Tract', 'Block Address', 'CIBRS Status', 'CIBRS Status Date', 'Domestic Violence Incident', 'BCS Area', 'Stolen Vehicles']


## Filtering columns
Out of these columns, we want to keep incident_uid, incident date, census_tract, zip_code, offense_description, and offense/CIBRS category. 

In [54]:
for i, col in enumerate(df.columns):
    print(i, col)

0 Incident UID
1 Agency
2 Incident Date
3 CIBRS Unique Offense ID
4 Crime Against Category
5 CIBRS Offense Code
6 CIBRS Offense Description
7 Victim UID
8 Victim Category
9 Victim Age
10 Victim Race
11 Victim Sex
12 Beat
13 City
14 Zip Code
15 Census Tract
16 Block Address
17 CIBRS Status
18 CIBRS Status Date
19 Domestic Violence Incident
20 BCS Area
21 Stolen Vehicles


Keep column 0, 2, 3, 4, 5, 14, 15

In [55]:
keep_col_index = [0, 2, 3, 4, 5, 6, 14, 15]
keep_col_names = []
for i, col in enumerate(df.columns):
    if i in keep_col_index:
        keep_col_names.append(col)

print(keep_col_names)

['Incident UID', 'Incident Date', 'CIBRS Unique Offense ID', 'Crime Against Category', 'CIBRS Offense Code', 'CIBRS Offense Description', 'Zip Code', 'Census Tract']


In [56]:
df_clean = df[keep_col_names].copy()
display(df_clean.head())

,Incident UID,Incident Date,CIBRS Unique Offense ID,Crime Against Category,CIBRS Offense Code,CIBRS Offense Description,Zip Code,Census Tract
0,604311,2021 Jan 01 01:25:00 AM,604311-290,Property,290,Destruction/Damage/Vandalism of Property,92115.0,NaN
1,605986,2021 Jan 01 08:40:00 AM,605986-26C,Property,26C,Impersonation,92101.0,NaN
2,773968,2021 Jan 01 12:30:00 PM,773968-13A-797918,People,13A,Aggravated Assault,NaN,NaN
3,687369,2021 Jan 03 01:00:00 AM,687369-23H,Property,23H,All Other Larceny,92019.0,NaN
4,1072215,2021 Jan 03 04:37:00 PM,1072215-13B-1109606,People,13B,Simple Assault,NaN,NaN


In [57]:
print("Total rows:", len(df_clean))
print("Unique incident IDS:", df_clean["Incident UID"].nunique())
print("Unique offense IDS:", df["CIBRS Unique Offense ID"].nunique())

Total rows: 806935
Unique incident IDS: 637013
Unique offense IDS: 752082


## Determining Patrol Offenses

In [58]:
display(df["Crime Against Category"].value_counts(dropna=False))

Crime Against Category
Property    475680
People      188204
Society     143051
Name: count, dtype: int64

In [59]:
display(df_clean["CIBRS Offense Description"].value_counts(dropna=False))

CIBRS Offense Description
Motor Vehicle Theft                           110388
Simple Assault                                106528
Destruction/Damage/Vandalism of Property       78089
Drug/Narcotic Violations                       74094
All Other Larceny                              64238
Theft From Motor Vehicle                       57112
Aggravated Assault                             52785
Drug Equipment Violations                      51317
Shoplifting                                    35470
Burglary/Breaking & Entering                   34720
Impersonation                                  18204
Weapon Law Violations                          15747
Intimidation                                   13918
Robbery                                        13168
Theft of Motor Vehicle Parts/Accessories       12276
Identity Theft                                 10862
Stolen Property Offenses                        9654
Theft From Building                             8629
False Pretenses/Swin

In [60]:
display(df_clean["CIBRS Offense Code"].value_counts().head(30))

CIBRS Offense Code
240    110388
13B    106528
290     78089
35A     74094
23H     64238
23F     57112
13A     52785
35B     51317
23C     35470
220     34720
26C     18204
520     15747
13C     13918
120     13168
23G     12276
26F     10862
280      9654
23D      8629
26A      8248
26B      5665
11D      5628
100      3887
250      3769
11A      3043
200      2175
40A      1226
270       958
11B       749
23B       714
23A       632
Name: count, dtype: int64

In [61]:
print("Data type:", df_clean["Incident Date"].dtype)
print("Missing dates:", df_clean["Incident Date"].isna().sum())
print("Earliest date:", df["Incident Date"].min())
print("Latest date:", df["Incident Date"].max())

Data type: object
Missing dates: 0
Earliest date: 2021 Apr 01 01:00:00 AM
Latest date: 2026 Sep 30 12:59:00 AM


In [62]:
df_clean["Incident Date"] = pd.to_datetime(df_clean["Incident Date"], format = "%Y %b %d %I:%M:%S %p", errors="coerce")
print("Data type:", df_clean["Incident Date"].dtype)


Data type: datetime64[ns]


In [63]:
df_clean["Year"] = df_clean["Incident Date"].dt.year
df_clean["Month"] = df_clean["Incident Date"].dt.month
df_clean["Day"] = df_clean["Incident Date"].dt.day
df_clean["Hour"] = df_clean["Incident Date"].dt.hour
df_clean["Minute"] = df_clean["Incident Date"].dt.minute

In [64]:
display(df_clean.head())

,Incident UID,Incident Date,CIBRS Unique Offense ID,Crime Against Category,CIBRS Offense Code,CIBRS Offense Description,Zip Code,Census Tract,Year,Month,Day,Hour,Minute
0,604311,2021-01-01 01:25:00,604311-290,Property,290,Destruction/Damage/Vandalism of Property,92115.0,NaN,2021,1,1,1,25
1,605986,2021-01-01 08:40:00,605986-26C,Property,26C,Impersonation,92101.0,NaN,2021,1,1,8,40
2,773968,2021-01-01 12:30:00,773968-13A-797918,People,13A,Aggravated Assault,NaN,NaN,2021,1,1,12,30
3,687369,2021-01-03 01:00:00,687369-23H,Property,23H,All Other Larceny,92019.0,NaN,2021,1,3,1,0
4,1072215,2021-01-03 16:37:00,1072215-13B-1109606,People,13B,Simple Assault,NaN,NaN,2021,1,3,16,37


In [65]:
offense_types = sorted(df_clean["CIBRS Offense Description"].dropna().unique())

for offense in offense_types:
    print(offense)

ANIMAL CRUELTY
Aggravated Assault
All Other Larceny
Arson
Assisting or Promoting Prostitution
Betting/Wagering
Bribery
Burglary/Breaking & Entering
Counterfeiting/Forgery
Credit Card/Automatic Teller Fraud
Destruction/Damage/Vandalism of Property
Drug Equipment Violations
Drug/Narcotic Violations
Embezzlement
Extortion/Blackmail
False Pretenses/Swindle/Confidence Game
Forcible Fondling
Forcible Rape
Forcible Sodomy
Gambling Equipment Violations
Hacking/Computer Invasion
Human Trafficking,Involuntary Servitude
Identity Theft
Impersonation
Incest
Intimidation
Justifiable Homicide
Kidnaping/Abduction
Motor Vehicle Theft
Murder and Nonnegligent Manslaughter
Negligent Manslaughter
Operating/Promoting/Assisting Gambling
Pocket-picking
Pornography/Obscene Material
Prostitution
Purse-snatching
Robbery
Sexual Assault With An Object
Shoplifting
Simple Assault
Statutory Rape
Stolen Property Offenses
Theft From Building
Theft From Coin Operated Machine or Device
Theft From Motor Vehicle
Theft of M

In [66]:
non_patrol_offenses = [
    "Counterfeiting/Forgery", 
    "Credit Card/Automatic Teller Fraud", 
    "Embezzlement", 
    "False Pretenses/Swindle/Confidence Game", 
    "Identity Theft", 
    "Impersonation", 
    "Hacking/Computer Invasion", 
    "Welfare Fraud", 
    "Betting/Wagering", 
    "Gambling Equipment Violations", 
    "Operating/Promoting/Assisting Gambling", 
    "Pornography/Obscene Material", 
    "Birbery"
]

In [67]:
df_patrol = df_clean[
    ~df_clean["CIBRS Offense Description"].isin(non_patrol_offenses)
].copy()
offense_types = sorted(df_patrol["CIBRS Offense Description"].dropna().unique())

for offense in offense_types:
    print(offense)


ANIMAL CRUELTY
Aggravated Assault
All Other Larceny
Arson
Assisting or Promoting Prostitution
Bribery
Burglary/Breaking & Entering
Destruction/Damage/Vandalism of Property
Drug Equipment Violations
Drug/Narcotic Violations
Extortion/Blackmail
Forcible Fondling
Forcible Rape
Forcible Sodomy
Human Trafficking,Involuntary Servitude
Incest
Intimidation
Justifiable Homicide
Kidnaping/Abduction
Motor Vehicle Theft
Murder and Nonnegligent Manslaughter
Negligent Manslaughter
Pocket-picking
Prostitution
Purse-snatching
Robbery
Sexual Assault With An Object
Shoplifting
Simple Assault
Statutory Rape
Stolen Property Offenses
Theft From Building
Theft From Coin Operated Machine or Device
Theft From Motor Vehicle
Theft of Motor Vehicle Parts/Accessories
Weapon Law Violations


## Determining time steps

In [68]:
INTERVAL_MINUTES = 15

df_patrol["Time Step"] = (
    df_patrol["Incident Date"].dt.hour * 60
    + df_patrol["Incident Date"].dt.minute
) // INTERVAL_MINUTES

df_fin = df_patrol.copy()

display(df_fin.head())

,Incident UID,Incident Date,CIBRS Unique Offense ID,Crime Against Category,CIBRS Offense Code,CIBRS Offense Description,Zip Code,Census Tract,Year,Month,Day,Hour,Minute,Time Step
0,604311,2021-01-01 01:25:00,604311-290,Property,290,Destruction/Damage/Vandalism of Property,92115.0,NaN,2021,1,1,1,25,5
2,773968,2021-01-01 12:30:00,773968-13A-797918,People,13A,Aggravated Assault,NaN,NaN,2021,1,1,12,30,50
3,687369,2021-01-03 01:00:00,687369-23H,Property,23H,All Other Larceny,92019.0,NaN,2021,1,3,1,0,4
4,1072215,2021-01-03 16:37:00,1072215-13B-1109606,People,13B,Simple Assault,NaN,NaN,2021,1,3,16,37,66
5,608584,2021-01-03 19:30:00,608584-240,Property,240,Motor Vehicle Theft,92110.0,68.02,2021,1,3,19,30,78


Ok so some things to consider. 
1. what offenses are we considering as patrol
2. should we get rid of duplicate ids? will we need to know the specific offenses if we're just counting patrol crimes?

And we should save final df as df_fin

## Mapping geography to unique coords

In [69]:
print("ZIP codes:", df_fin["Zip Code"].nunique())
print("Census tracts:", df_fin["Census Tract"].nunique())

ZIP codes: 112
Census tracts: 735


San Diego Boundaries: https://www.sandiegocounty.gov/content/sdc/bhs/county-zip-codes-regional-boundaries.html 

Since this is also sourced from SANDAG data from 2025, this should align enough with our crime dataset. 

1. North (North Central + North Inland + North Coastal) 
2. Central 
3. South 
4. East 

Roughly 10 random zip codes from each 

In [70]:
import random 
random.seed(42)

region_dict = {
    "North": [92037, 92093, 92106, 92107, 92108, 92109, 
              92110, 92111, 92117, 92119, 92120, 92121, 
              92122, 92123, 92124, 92126, 92130, 92131, 
              92140, 92145, 92161,92007, 92008, 92009, 
              92010, 92011, 92014, 92024, 92054, 92055, 
              92056, 92057, 92058, 92067, 92075, 92081, 
              92083, 92084, 92091, 92672, 92003, 92004,
              92025, 92026, 92027, 92028, 92029, 92036, 
              92059, 92060, 92061, 92064, 92065, 92066, 
              92069, 92070, 92078, 92082, 92086, 92096, 
              92127, 92128, 92129, 92259, 92536], 
    "South": [91902, 91910, 91911, 91913, 91914, 91915, 
              91932, 91950, 92118, 92135, 92154, 92155, 92173], 
    "Central": [92101, 92102, 92103, 92104, 92105, 92113 , 
                92114, 92115, 92116, 92134, 92136, 92139, 92182], 
    "East": [91901, 91905, 91906, 91916, 91917, 91931, 91934, 
             91935, 91941, 91942, 91945, 91948, 91962, 91963, 
             91977, 91978, 91980, 92019, 92020, 92021, 92040, 92071]
}

sd_zips = set(df_fin["Zip Code"].dropna().astype(int).unique())
list(sd_zips)[:5]

[np.int64(92672),
 np.int64(92161),
 np.int64(92173),
 np.int64(92182),
 np.int64(91901)]

In [71]:
selected_zips = {}

for region, zips in region_dict.items():
    selected_zips[region] = random.sample(zips, 10)

print(selected_zips)

all_select_zips = [
    zip_code 
    for zips in selected_zips.values()
    for zip_code in zips 
]

{'North': [92124, 92107, 92036, 92131, 92126, 92536, 92117, 92129, 92110, 92026], 'South': [92155, 92118, 91910, 92135, 91932, 91902, 91950, 92173, 92154, 91914], 'Central': [92116, 92134, 92101, 92182, 92104, 92114, 92102, 92139, 92105, 92103], 'East': [91901, 91931, 91963, 91945, 91941, 91917, 91934, 91962, 92040, 91905]}


In [72]:
#check if zips are in the sandag crime data 
for region, zips in selected_zips.items():
    for zip_code in zips:
        if zip_code not in sd_zips:
            print("Invalid zip code")

In [73]:
#filter df_fin by selectd zip codes 
df_fin2 = df_fin[df_fin["Zip Code"].isin(all_select_zips)].copy() 
display(df_fin2.head())

,Incident UID,Incident Date,CIBRS Unique Offense ID,Crime Against Category,CIBRS Offense Code,CIBRS Offense Description,Zip Code,Census Tract,Year,Month,Day,Hour,Minute,Time Step
5,608584,2021-01-03 19:30:00,608584-240,Property,240,Motor Vehicle Theft,92110.0,68.02,2021,1,3,19,30,78
6,1002702,2021-01-03 20:00:00,1002702-240,Property,240,Motor Vehicle Theft,92102.0,41.02,2021,1,3,20,0,80
9,606017,2021-01-05 13:35:00,606017-23F,Property,23F,Theft From Motor Vehicle,92154.0,NaN,2021,1,5,13,35,54
10,642766,2021-01-05 16:00:00,642766-240,Property,240,Motor Vehicle Theft,91910.0,134.09,2021,1,5,16,0,64
14,1071423,2021-01-07 14:40:00,1071423-120,Property,120,Robbery,92110.0,66.00,2021,1,7,14,40,58


In [74]:
unique_tracts = df_fin2["Census Tract"].dropna().unique()
print("Number of unique tracts: ", len(unique_tracts))

Number of unique tracts:  420
